# LeetCode 606 — Construct String from Binary Tree bằng Python

Link đề: https://leetcode.com/problems/construct-string-from-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Tree, Depth-First Search, String

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        ...
```

Ghi chú: notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (`[1,2,3,null,4]`). Ví dụ 3 và 4 (một nút; chuỗi chỉ có con phải) là do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: biến một cây nhị phân thành **chuỗi có ngoặc** theo thứ tự **tiền thứ tự (preorder)**, trong đó nhiều cặp ngoặc rỗng `()` bị **lược bỏ** — trừ trường hợp nút **có con phải nhưng không có con trái**.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, `f-string`, `str(x)` cho số âm, nối chuỗi bằng `"".join(list)`, đệ quy trả về chuỗi, và **ngăn xếp (stack)** bằng `list` để thay đệ quy.
3. Đi từ **dựng chuỗi đầy đủ rồi xóa `()` thừa bằng regex** → **đệ quy 3 trường hợp** → **đệ quy ghi vào một list dùng chung** → **ngăn xếp lưu cả nút lẫn ký tự ngoặc** → **ngăn xếp có tập "đã thăm" để đóng ngoặc**.
4. Hiểu mấu chốt: quy tắc bỏ `()` **không đối xứng** — thiếu con **phải** thì bỏ được, thiếu con **trái** thì **phải giữ** (nếu có con phải) để chuỗi còn **đọc ngược lại được đúng cây**.
5. Tránh các lỗi kinh điển: bỏ mọi `()` (mất thông tin con trái), giữ mọi `()` (thừa), quên `str()` cho `val`, và quên rằng cây `10^4` nút có thể **sâu `10^4`** làm đệ quy tràn.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân, hãy tạo **chuỗi biểu diễn cây** theo duyệt **tiền thứ tự** với các quy tắc:

- **Nút**: mỗi nút được biểu diễn bằng **giá trị nguyên** của nó.
- **Ngoặc cho con**: nếu nút có **ít nhất một con**, các con được bọc trong ngoặc. Con **trái** (nếu có) nằm trong cặp ngoặc đứng **ngay sau** giá trị nút; con **phải** (nếu có) nằm trong cặp ngoặc **sau** cặp của con trái.
- **Bỏ ngoặc rỗng**: mọi cặp `()` rỗng đều bị bỏ, **trừ một ngoại lệ**: nút **có con phải nhưng không có con trái** thì phải viết `()` đứng trước con phải, để biết con trái đang vắng mặt.

Tóm lại: thiếu **phải** hay thiếu **cả hai** thì bỏ `()`; thiếu **trái** nhưng có phải thì **giữ** `()`.

**Ví dụ 1**

- Input: `root = [1,2,3,4]`
- Output: `"1(2(4))(3)"`
- Giải thích: ban đầu là `"1(2(4)())(3()())"`; bỏ mọi cặp rỗng được `"1(2(4))(3)"`.

**Ví dụ 2**

- Input: `root = [1,2,3,null,4]`
- Output: `"1(2()(4))(3)"`
- Giải thích: giống ví dụ 1, nhưng `()` sau `2` là **bắt buộc** vì `2` không có con trái mà có con phải `4`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [1]`
- Output: `"1"`

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [-1,null,-2]`
- Output: `"-1()(-2)"`

**Ràng buộc:**

- Số nút trong khoảng `[1, 10^4]`.
- `-1000 <= Node.val <= 1000`.

Chú ý: giá trị có thể **âm** và có thể là `0`; chuỗi có thể dài vài chục nghìn ký tự và cây có thể **rất sâu** (một dây dài `10^4` nút).

In [1]:
from typing import Optional, List
from collections import deque
import random
import re
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)


examples = [
    ([1, 2, 3, 4], "1(2(4))(3)"),
    ([1, 2, 3, None, 4], "1(2()(4))(3)"),
    ([1], "1"),
    ([-1, None, -2], "-1()(-2)"),
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> kỳ vọng", repr(expected))

[1, 2, 3, 4] -> kỳ vọng '1(2(4))(3)'
[1, 2, 3, None, 4] -> kỳ vọng '1(2()(4))(3)'
[1] -> kỳ vọng '1'
[-1, None, -2] -> kỳ vọng '-1()(-2)'


## 3. Kiến thức nền

### 3.1. Nút cây, `build_tree` và `tree_to_list`

Mỗi nút `TreeNode` có `val`, `left`, `right` (con trống là `None`). LeetCode viết cây thành list **theo từng hàng, từ trái sang phải**, `None` là chỗ trống. `build_tree` dựng cây bằng hàng đợi; `tree_to_list` làm ngược lại. Cây `[1,2,3,None,4]`:

```
      1
     / \
    2   3
     \
      4
```

In [2]:
t = build_tree([1, 2, 3, None, 4])
print("gốc:", t.val, "| con trái:", t.left.val, "| con phải:", t.right.val)
print("con trái của 2:", t.left.left, "| con phải của 2:", t.left.right.val)
print(tree_to_list(t))

gốc: 1 | con trái: 2 | con phải: 3
con trái của 2: None | con phải của 2: 4
[1, 2, 3, None, 4]


### 3.2. Duyệt tiền thứ tự (preorder)

**Preorder** = thăm **nút trước**, rồi con **trái**, rồi con **phải**. Chuỗi của bài này đi theo đúng thứ tự đó: giá trị nút xuất hiện trước giá trị các con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)
    preorder(node.left, out)
    preorder(node.right, out)

res = []
preorder(build_tree([1, 2, 3, 4, None, None, 5]), res)
print(res)          # 1, 2, 4, 3, 5

[1, 2, 4, 3, 5]


### 3.3. `f-string` và `str()` với số âm

`f"{x}"` đổi `x` thành chuỗi, kể cả số âm (`"-5"`) và `0`. Nối thẳng số với chuỗi bằng `+` gây `TypeError`, nên luôn dùng `str(val)` hoặc f-string.

In [4]:
val = -12
print(f"{val}({val + 2})")
print(str(0), str(-7), str(1000))
try:
    print(val + "(")
except TypeError as e:
    print("lỗi:", e)

-12(-10)
0 -7 1000
lỗi: unsupported operand type(s) for +: 'int' and 'str'


### 3.4. Đệ quy trả về chuỗi

Một hàm đệ quy có thể **trả về chuỗi** của cây con, rồi cha ghép chúng lại. Mẫu chung: điều kiện dừng cho `None`, rồi ghép `giá trị + kết quả con trái + kết quả con phải`.

In [5]:
def height(node):
    return 0 if node is None else 1 + max(height(node.left), height(node.right))

def describe(node):
    if node is None:
        return "·"
    return f"[{node.val} {describe(node.left)} {describe(node.right)}]"

t = build_tree([1, 2, 3, None, 4])
print(describe(t), "| chiều cao =", height(t))

[1 [2 · [4 · ·]] [3 · ·]] | chiều cao = 3


### 3.5. Chuỗi là bất biến: nối nhiều lần vs `"".join`

Chuỗi trong Python **bất biến**: `s += x` tạo chuỗi mới, sao chép toàn bộ nội dung cũ. Nối nhiều lần trong vòng lặp dài có thể tốn `O(n²)`. Cách chuẩn: gom các mảnh vào **list** rồi gọi `"".join(list)` đúng một lần (`O(n)`).

In [6]:
pieces = []
for i in range(5):
    pieces.append(str(i))
    pieces.append(",")
print("".join(pieces))

parts = ["1", "(", "2", ")", "(", "3", ")"]
print("".join(parts))

0,1,2,3,4,
1(2)(3)


### 3.6. Giới hạn đệ quy và ngăn xếp (stack)

Python mặc định chỉ cho đệ quy sâu khoảng `1000` tầng. Cây `10^4` nút dạng **một dây** sâu `10^4` tầng ⇒ có thể gặp `RecursionError`. Cách tránh: tự làm **ngăn xếp** bằng `list` (`append` = đẩy, `pop()` = lấy ra, vào sau ra trước — LIFO) để **thay đệ quy** bằng vòng `while`.

In [7]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
stack = []
stack.append("a")
stack.append("b")
stack.append("c")
print("pop ->", stack.pop(), "| còn", stack)      # lấy phần tử đẩy vào SAU CÙNG

giới hạn đệ quy mặc định: 1000
pop -> c | còn ['a', 'b']


### 3.7. So sánh với `None`: dùng `is None`

Nút con rỗng là `None`. Viết `if node.left:` cũng chạy được vì đối tượng `TreeNode` luôn "đúng", nhưng để rõ nghĩa ta dùng `is None` / `is not None`. (Lưu ý: **đừng** viết `if node.val:` để kiểm tra có nút hay không vì `val = 0` là "sai" trong Python!)

In [8]:
node = TreeNode(0)
print("val = 0 -> bool(node.val) =", bool(node.val), "| nút có tồn tại:", node is not None)
print("node.left is None:", node.left is None)

val = 0 -> bool(node.val) = False | nút có tồn tại: True
node.left is None: True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét cây `[1,2,3,None,4]` (ví dụ 2). Ta viết **đầy đủ** mọi cặp ngoặc trước: mỗi nút `x` đi kèm **hai nhóm ngoặc**, nhóm trái rồi nhóm phải, nhóm trống thì là `()`:

- Nút `4` (lá): `4()()`
- Nút `2` (không trái, phải là `4`): `2` + `()` + `(4()())` = `2()(4()())`
- Nút `3` (lá): `3()()`
- Nút `1`: `1` + `(2()(4()()))` + `(3()())`

Đầy đủ: `1(2()(4()()))(3()())`.

Giờ **bỏ ngoặc rỗng thừa**:

- Nút `4` là lá: bỏ cả hai `()` ⇒ `4`.
- Nút `3` là lá: bỏ cả hai ⇒ `3`.
- Nút `2`: nhóm trái `()` **phải giữ** (có con phải `4`); không có gì sau con phải ⇒ `2()(4)`.
- Nút `1`: ghép ⇒ `1(2()(4))(3)` ✓.

**Quy tắc cho một nút `x`** (3 trường hợp):

1. **Lá** (không con): chỉ `x`.
2. **Chỉ có con trái hoặc không có con phải**: `x(trái)` — nhóm phải rỗng bị bỏ.
3. **Có con phải**: `x(trái)(phải)` — nếu trái vắng thì `(trái)` là `()`.

Vì sao `()` ở trường hợp 3 là **bắt buộc**? Nếu bỏ: `1(2(4))(3)` có thể đọc ra `2` là con trái của `1` và `4` là con trái của `2` — **không phân biệt được** với cây trong ví dụ 1. Ngoặc rỗng cho ta biết "con trái đang vắng", khiến chuỗi **một-một** với cây. Hàm dưới in từng bước viết chuỗi **đầy đủ** rồi chuỗi **rút gọn** để bạn thấy tận mắt.

In [9]:
def full_form(node):
    """Mỗi nút kèm đủ hai nhóm ngoặc; con vắng là nhóm rỗng '()'."""
    if node is None:
        return ""
    return f"{node.val}({full_form(node.left)})({full_form(node.right)})"

for vals, expected in examples:
    print(f"{str(vals):28s} đầy đủ: {full_form(build_tree(vals)):28s} -> rút gọn đúng: {expected}")

[1, 2, 3, 4]                 đầy đủ: 1(2(4()())())(3()())         -> rút gọn đúng: 1(2(4))(3)
[1, 2, 3, None, 4]           đầy đủ: 1(2()(4()()))(3()())         -> rút gọn đúng: 1(2()(4))(3)
[1]                          đầy đủ: 1()()                        -> rút gọn đúng: 1
[-1, None, -2]               đầy đủ: -1()(-2()())                 -> rút gọn đúng: -1()(-2)


## 5. Cách 1 — Brute Force: viết đầy đủ rồi xóa `()` thừa

**Ý tưởng:** làm đúng hai bước như tính tay. Bước 1: dựng chuỗi **đầy đủ** (mỗi nút kèm đủ hai nhóm ngoặc). Bước 2: **xóa** các cặp `()` **không cần thiết**.

Khi nào một `()` **cần**? Chỉ khi nó là nhóm trái rỗng của nút có con phải — tức là ngay sau nó là `(` rồi **một chữ số hoặc dấu `-`** (bắt đầu giá trị của một con thật). Mọi `()` khác (nhóm trái rỗng của lá, nhóm phải rỗng) đều xóa. Ta dùng biểu thức chính quy `re.sub(r'\(\)(?!\([-0-9])', '', s)`: xóa mọi `()` **không** được theo sau bởi `(` + `-`/chữ số. Mọi quyết định đưa ra trên chuỗi **gốc**, nên việc xóa cặp này không ảnh hưởng cặp khác.

In [10]:
def tree2str_bruteforce(root):
    full = full_form(root)
    return re.sub(r"\(\)(?!\([-0-9])", "", full)

for vals, expected in examples:
    got = tree2str_bruteforce(build_tree(vals))
    print(vals, repr(got), "OK" if got == expected else "SAI")

[1, 2, 3, 4] '1(2(4))(3)' OK
[1, 2, 3, None, 4] '1(2()(4))(3)' OK
[1] '1' OK
[-1, None, -2] '-1()(-2)' OK


**Độ phức tạp:** dựng chuỗi đầy đủ bằng đệ quy với nối chuỗi và duyệt regex: `O(n)` về mặt số nút, nhưng nối chuỗi lồng nhau có thể tốn thêm (cây một dây: `O(n²)` ký tự sao chép). Thêm bộ nhớ cho chuỗi đầy đủ (dài hơn chuỗi đáp án).

**Nút thắt:** phải dựng một chuỗi **thừa** rồi mới lọc; regex "đoán" nhu cầu `()` từ ký tự kế tiếp — khó đọc và dễ sai khi mở rộng. Ta muốn **quyết định ngay** lúc đi qua từng nút xem có cần viết `()` hay không.

In [11]:
# Độ dài chuỗi đầy đủ so với chuỗi rút gọn trên cây ngẫu nhiên: phần "thừa" lớn thế nào?
rng = random.Random(1)
for n in [10, 100, 1000]:
    t = build_tree(random_tree_list(rng, n, -9, 9))
    full = full_form(t)
    short = tree2str_bruteforce(t)
    print(f"n = {n:4d}: đầy đủ {len(full):5d} ký tự | rút gọn {len(short):5d} ký tự | thừa {len(full) - len(short):5d}")

n =   10: đầy đủ    55 ký tự | rút gọn    39 ký tự | thừa    16
n =  100: đầy đủ   546 ký tự | rút gọn   360 ký tự | thừa   186
n = 1000: đầy đủ  5472 ký tự | rút gọn  3818 ký tự | thừa  1654


## 6. Ý tưởng thuật toán / bước đột phá

Hãy để mỗi nút **tự quyết định** nó cần viết gì, chỉ dựa vào hai nhận xét về con của nó:

| `left` | `right` | Chuỗi của nút |
|---|---|---|
| `None` | `None` | `x` |
| có | `None` | `x(L)` |
| có | có | `x(L)(R)` |
| `None` | có | `x()(R)` |

Dòng thứ ba và thứ tư gộp được: **nếu có `right`** thì `x(L)(R)` với `L` là chuỗi rỗng khi trái vắng (nên `(L)` thành `()`). Và gộp dòng 1-2: **nếu không có `right`** thì `x(L)` khi có trái, `x` khi cũng không có trái.

Do đó chỉ cần **3 trường hợp** theo đúng thứ tự kiểm tra (cũng là lời giải gọi là chuẩn):

1. `left is None and right is None` ⇒ `str(val)`.
2. `right is None` ⇒ `f"{val}({dfs(left)})"`.
3. còn lại (có `right`) ⇒ `f"{val}({dfs(left)})({dfs(right)})"`, với `dfs(None) = ""`.

Điểm then chốt: `dfs(None)` trả **chuỗi rỗng**, nên khi trái vắng mà phải có, `({dfs(left)})` tự nhiên thành `()` — không cần nhánh riêng. Mỗi nút được ghé **đúng một lần** ⇒ `O(n)`.

## 7. Các cách tiếp theo

Cùng một quy tắc 3 trường hợp, ta có nhiều cách thực hiện: **đệ quy ghép chuỗi**, **đệ quy ghi vào một list dùng chung**, và hai cách **không đệ quy** dùng ngăn xếp (ngăn xếp chứa cả nút lẫn ký tự ngoặc; ngăn xếp kèm tập "đã thăm"). Khi nộp LeetCode, chỉ đổi tên class thành `Solution`.

### 7.1. Cách 2 — Đệ quy 3 trường hợp (ghép chuỗi)

**Giải thích từng đoạn:**

1. `dfs(None)` trả `""`.
2. Nút là lá ⇒ `str(val)`.
3. Không có con phải ⇒ chỉ bọc con trái: `f"{val}({dfs(left)})"`.
4. Có con phải ⇒ bọc cả hai: `f"{val}({dfs(left)})({dfs(right)})"` (trái vắng ⇒ `()`).

In [12]:
class SolutionRecursive:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 40000))

        def dfs(node):
            if node is None:
                return ""
            if node.left is None and node.right is None:
                return str(node.val)
            if node.right is None:
                return f"{node.val}({dfs(node.left)})"
            return f"{node.val}({dfs(node.left)})({dfs(node.right)})"

        return dfs(root)

for vals, expected in examples:
    print(vals, repr(SolutionRecursive().tree2str(build_tree(vals))), "kỳ vọng", repr(expected))

[1, 2, 3, 4] '1(2(4))(3)' kỳ vọng '1(2(4))(3)'
[1, 2, 3, None, 4] '1(2()(4))(3)' kỳ vọng '1(2()(4))(3)'
[1] '1' kỳ vọng '1'
[-1, None, -2] '-1()(-2)' kỳ vọng '-1()(-2)'


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        def dfs(root):
            if root is None:
                return ''
            if root.left is None and root.right is None:
                return str(root.val)
            if root.right is None:
                return f'{root.val}({dfs(root.left)})'
            return f'{root.val}({dfs(root.left)})({dfs(root.right)})'

        return dfs(root)
```

**Độ phức tạp:** mỗi nút xử lý một lần; nhưng mỗi cha **sao chép** chuỗi của con vào chuỗi mới. Với cây một dây dài `n` thì tổng sao chép `O(n²)` ký tự (chỉ là `memcpy` nhanh với `n ≤ 10^4`). Bộ nhớ: `O(h)` ngăn xếp đệ quy với `h` là chiều cao (tới `n`).

### 7.2. Cách 3 — Đệ quy ghi vào một list dùng chung

**Ý tưởng:** thay vì mỗi tầng trả về một chuỗi rồi ghép (sao chép nhiều lần), ta để mọi tầng **ghi trực tiếp vào cùng một list `parts`** theo thứ tự xuất hiện, cuối cùng `"".join(parts)` **một lần**.

**Giải thích từng đoạn:**

1. Thăm nút: `parts.append(str(val))`.
2. Nếu là lá thì xong.
3. Ngược lại mở nhóm trái: `parts.append("(")`; nếu có trái thì đệ quy; đóng `")"`. (Trái vắng ⇒ ngoặc rỗng `()`, đúng và cần thiết **chỉ khi có phải** — ta kiểm tra điều đó ngay sau.)
4. Nếu có phải thì mở `"("`, đệ quy phải, đóng `")"`. Nếu **không có phải** mà trái vắng thì đã bị loại ở bước 2 (vì khi đó là lá).

Chú ý ở bước 3: khi nút **không phải lá**, hoặc có trái, hoặc có phải; nếu chỉ có phải thì ngoặc trái rỗng `()` được in ra đúng như yêu cầu; nếu chỉ có trái thì in `(trái)` rồi không in nhóm phải.

In [13]:
class SolutionSharedList:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 40000))
        parts = []

        def dfs(node):
            parts.append(str(node.val))
            if node.left is None and node.right is None:
                return
            parts.append("(")
            if node.left is not None:
                dfs(node.left)
            parts.append(")")
            if node.right is not None:
                parts.append("(")
                dfs(node.right)
                parts.append(")")

        if root is not None:
            dfs(root)
        return "".join(parts)

for vals, expected in examples:
    print(vals, repr(SolutionSharedList().tree2str(build_tree(vals))), "kỳ vọng", repr(expected))

[1, 2, 3, 4] '1(2(4))(3)' kỳ vọng '1(2(4))(3)'
[1, 2, 3, None, 4] '1(2()(4))(3)' kỳ vọng '1(2()(4))(3)'
[1] '1' kỳ vọng '1'
[-1, None, -2] '-1()(-2)' kỳ vọng '-1()(-2)'


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        parts = []

        def dfs(node):
            parts.append(str(node.val))
            if node.left is None and node.right is None:
                return
            parts.append("(")
            if node.left is not None:
                dfs(node.left)
            parts.append(")")
            if node.right is not None:
                parts.append("(")
                dfs(node.right)
                parts.append(")")

        dfs(root)
        return "".join(parts)
```

**Độ phức tạp:** thời gian `O(n)` (mỗi nút một lần, mỗi ký tự ghi một lần, không sao chép lồng nhau); bộ nhớ `O(n)` cho `parts` (chính là đáp án) cộng `O(h)` ngăn xếp đệ quy.

### 7.3. Cách 4 — Ngăn xếp chứa cả nút lẫn ký tự ngoặc (không đệ quy)

**Ý tưởng:** ta muốn in theo thứ tự `giá trị, "(", [trái], ")", "(", [phải], ")"`. Trong ngăn xếp (LIFO), thứ tự đẩy vào phải **ngược** thứ tự mong muốn in ra. Ta cho ngăn xếp chứa **hai loại phần tử**: **nút** (cần xử lý) và **chuỗi** (ký tự ngoặc có sẵn, chỉ việc in ra).

**Giải thích từng đoạn:**

1. Lấy phần tử `x` trên đỉnh. Nếu `x` là chuỗi thì ghi nó vào kết quả; nếu `None` thì bỏ qua.
2. Nếu `x` là nút: ghi `str(val)`.
3. **Đẩy nhóm phải trước** (vì sẽ in sau): nếu có phải thì đẩy `")"`, `right`, `"("` (đẩy theo thứ tự này để khi lấy ra được `"("`, `right`, `")"`).
4. **Rồi đẩy nhóm trái** (in trước): nếu có trái đẩy `")"`, `left`, `"("`; nếu trái vắng **nhưng có phải** đẩy `")"`, `"("` để in ra `()`.

In [14]:
class SolutionStackTokens:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        res = []
        stack = [root]
        while stack:
            x = stack.pop()
            if isinstance(x, str):
                res.append(x)
                continue
            if x is None:
                continue
            res.append(str(x.val))
            if x.right is not None:
                stack.extend([")", x.right, "("])      # nhóm phải: in sau nên đẩy trước
            if x.left is not None:
                stack.extend([")", x.left, "("])       # nhóm trái
            elif x.right is not None:
                stack.extend([")", "("])               # trái vắng nhưng có phải: in "()"
        return "".join(res)

for vals, expected in examples:
    print(vals, repr(SolutionStackTokens().tree2str(build_tree(vals))), "kỳ vọng", repr(expected))

[1, 2, 3, 4] '1(2(4))(3)' kỳ vọng '1(2(4))(3)'
[1, 2, 3, None, 4] '1(2()(4))(3)' kỳ vọng '1(2()(4))(3)'
[1] '1' kỳ vọng '1'
[-1, None, -2] '-1()(-2)' kỳ vọng '-1()(-2)'


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        res = []
        stack = [root]
        while stack:
            x = stack.pop()
            if isinstance(x, str):
                res.append(x)
                continue
            if x is None:
                continue
            res.append(str(x.val))
            if x.right is not None:
                stack.extend([")", x.right, "("])
            if x.left is not None:
                stack.extend([")", x.left, "("])
            elif x.right is not None:
                stack.extend([")", "("])
        return "".join(res)
```

**Độ phức tạp:** thời gian `O(n)`, bộ nhớ `O(n)` (kết quả) và ngăn xếp `O(h)`-`O(n)`. **Không đệ quy** nên miễn nhiễm với `RecursionError` ở cây sâu `10^4`.

### 7.4. Cách 5 — Ngăn xếp nút + tập "đã thăm" để biết lúc đóng ngoặc

**Ý tưởng khác:** chỉ đẩy **nút** vào ngăn xếp; nút xuất hiện ở đỉnh **lần đầu** thì "mở" nó (in `"(" + val`) và đẩy con; khi nút xuất hiện ở đỉnh **lần thứ hai** (mọi con đã xử lý xong) thì "đóng" nó (in `")"`). Ta nhớ nút nào đã thăm bằng một `set`. Mỗi nút luôn được bọc trong một cặp ngoặc, nên ta bỏ cặp ngoặc **ngoài cùng** của gốc ở cuối bằng `[1:-1]`.

**Giải thích từng đoạn:**

1. `stack = [root]`, `visited = set()`.
2. Nhìn đỉnh `node = stack[-1]` (chưa lấy ra). Nếu đã thăm: `pop()` và in `")"`.
3. Nếu chưa: đánh dấu đã thăm, in `"(" + str(val)`; nếu **trái vắng, phải có** in thêm `"()"`.
4. Đẩy con **phải** trước rồi con **trái** (để trái được xử lý trước).
5. Cuối cùng bỏ ký tự đầu và cuối.

In [15]:
class SolutionVisited:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        if root is None:
            return ""
        res = []
        stack = [root]
        visited = set()
        while stack:
            node = stack[-1]
            if node in visited:
                stack.pop()
                res.append(")")
            else:
                visited.add(node)
                res.append("(" + str(node.val))
                if node.left is None and node.right is not None:
                    res.append("()")
                if node.right is not None:
                    stack.append(node.right)
                if node.left is not None:
                    stack.append(node.left)
        return "".join(res)[1:-1]

for vals, expected in examples:
    print(vals, repr(SolutionVisited().tree2str(build_tree(vals))), "kỳ vọng", repr(expected))

[1, 2, 3, 4] '1(2(4))(3)' kỳ vọng '1(2(4))(3)'
[1, 2, 3, None, 4] '1(2()(4))(3)' kỳ vọng '1(2()(4))(3)'
[1] '1' kỳ vọng '1'
[-1, None, -2] '-1()(-2)' kỳ vọng '-1()(-2)'


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        if root is None:
            return ""
        res = []
        stack = [root]
        visited = set()
        while stack:
            node = stack[-1]
            if node in visited:
                stack.pop()
                res.append(")")
            else:
                visited.add(node)
                res.append("(" + str(node.val))
                if node.left is None and node.right is not None:
                    res.append("()")
                if node.right is not None:
                    stack.append(node.right)
                if node.left is not None:
                    stack.append(node.left)
        return "".join(res)[1:-1]
```

**Độ phức tạp:** thời gian `O(n)`, bộ nhớ `O(n)`. Mỗi nút được ghé **hai lần** (mở và đóng), tổng vẫn tuyến tính.

Cách 4 "lên lịch trước" cả ký tự ngoặc; cách 5 "hoãn" việc đóng ngoặc cho tới khi quay lại nút. Cả hai đều mô phỏng đúng ngăn xếp gọi hàm của đệ quy.

## 8. TRACE từng bước

Ta theo dõi cách 2 (đệ quy) trên ví dụ 2 `[1,2,3,None,4]`: mỗi lời gọi in ra nút đang xét, trường hợp nào được dùng, và chuỗi nó trả về (thụt lề theo độ sâu).

Bảng trạng thái kết quả của từng nút (dòng cuối là đáp án):

| nút | trái | phải | trường hợp | chuỗi trả về |
|---|---|---|---|---|
| `4` | None | None | lá | `4` |
| `2` | None | `4` | có phải | `2()(4)` |
| `3` | None | None | lá | `3` |
| `1` | `2` | `3` | có phải | `1(2()(4))(3)` |

In [16]:
def trace_recursive(node, depth=0):
    pad = "  " * depth
    if node is None:
        print(f"{pad}dfs(None) -> ''")
        return ""
    print(f"{pad}dfs({node.val})  trái = {node.left.val if node.left else None}, phải = {node.right.val if node.right else None}")
    if node.left is None and node.right is None:
        r = str(node.val)
        case = "lá"
    elif node.right is None:
        r = f"{node.val}({trace_recursive(node.left, depth + 1)})"
        case = "chỉ có trái"
    else:
        left_s = trace_recursive(node.left, depth + 1)
        right_s = trace_recursive(node.right, depth + 1)
        r = f"{node.val}({left_s})({right_s})"
        case = "có phải"
    print(f"{pad}=> nút {node.val} [{case}] trả về {r!r}")
    return r

print("KẾT QUẢ:", repr(trace_recursive(build_tree([1, 2, 3, None, 4]))))

dfs(1)  trái = 2, phải = 3
  dfs(2)  trái = None, phải = 4
    dfs(None) -> ''
    dfs(4)  trái = None, phải = None
    => nút 4 [lá] trả về '4'
  => nút 2 [có phải] trả về '2()(4)'
  dfs(3)  trái = None, phải = None
  => nút 3 [lá] trả về '3'
=> nút 1 [có phải] trả về '1(2()(4))(3)'
KẾT QUẢ: '1(2()(4))(3)'


Quan sát: với nút `2`, `dfs(None)` trả `''`, nên `f"{val}({left_s})({right_s})"` thành `2()(4)` — ngoặc trái rỗng **tự xuất hiện** mà không cần nhánh riêng.

Giờ trace cách 4 (ngăn xếp chứa cả nút lẫn ngoặc): in ngăn xếp và chuỗi kết quả sau mỗi lần `pop`.

In [17]:
def trace_stack_tokens(root):
    def show(x):
        return x if isinstance(x, str) else ("None" if x is None else f"N{x.val}")
    res = []
    stack = [root]
    step = 0
    while stack:
        x = stack.pop()
        step += 1
        action = ""
        if isinstance(x, str):
            res.append(x)
            action = f"ghi ký tự {x!r}"
        elif x is None:
            action = "bỏ qua None"
        else:
            res.append(str(x.val))
            action = f"ghi giá trị {x.val}"
            if x.right is not None:
                stack.extend([")", x.right, "("])
                action += ", đẩy nhóm phải"
            if x.left is not None:
                stack.extend([")", x.left, "("])
                action += ", đẩy nhóm trái"
            elif x.right is not None:
                stack.extend([")", "("])
                action += ", đẩy '()' (trái vắng)"
        print(f"bước {step:2d}: {action:44s} | stack = {[show(s) for s in stack]} | kết quả = {''.join(res)!r}")
    return "".join(res)

print("KẾT QUẢ:", repr(trace_stack_tokens(build_tree([1, 2, 3, None, 4]))))

bước  1: ghi giá trị 1, đẩy nhóm phải, đẩy nhóm trái  | stack = [')', 'N3', '(', ')', 'N2', '('] | kết quả = '1'
bước  2: ghi ký tự '('                                | stack = [')', 'N3', '(', ')', 'N2'] | kết quả = '1('
bước  3: ghi giá trị 2, đẩy nhóm phải, đẩy '()' (trái vắng) | stack = [')', 'N3', '(', ')', ')', 'N4', '(', ')', '('] | kết quả = '1(2'
bước  4: ghi ký tự '('                                | stack = [')', 'N3', '(', ')', ')', 'N4', '(', ')'] | kết quả = '1(2('
bước  5: ghi ký tự ')'                                | stack = [')', 'N3', '(', ')', ')', 'N4', '('] | kết quả = '1(2()'
bước  6: ghi ký tự '('                                | stack = [')', 'N3', '(', ')', ')', 'N4'] | kết quả = '1(2()('
bước  7: ghi giá trị 4                                | stack = [')', 'N3', '(', ')', ')'] | kết quả = '1(2()(4'
bước  8: ghi ký tự ')'                                | stack = [')', 'N3', '(', ')'] | kết quả = '1(2()(4)'
bước  9: ghi ký tự ')'                                

## 9. Code nộp LeetCode

Ta chọn **Cách 2 (đệ quy 3 trường hợp)**: ngắn gọn nhất, bám sát quy tắc của đề, và đủ nhanh với `n ≤ 10^4` (LeetCode cho phép đệ quy sâu cho bài này). Nếu muốn an toàn tuyệt đối với cây rất sâu, dùng cách 4. Class `Solution` đúng chữ ký:

```python
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        def dfs(root):
            if root is None:
                return ''
            if root.left is None and root.right is None:
                return str(root.val)
            if root.right is None:
                return f'{root.val}({dfs(root.left)})'
            return f'{root.val}({dfs(root.left)})({dfs(root.right)})'

        return dfs(root)
```

In [18]:
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        def dfs(root):
            if root is None:
                return ''
            if root.left is None and root.right is None:
                return str(root.val)
            if root.right is None:
                return f'{root.val}({dfs(root.left)})'
            return f'{root.val}({dfs(root.left)})({dfs(root.right)})'

        return dfs(root)

for vals, expected in examples:
    got = Solution().tree2str(build_tree(vals))
    assert got == expected, (vals, got, expected)
    print(vals, "->", repr(got), "OK")

[1, 2, 3, 4] -> '1(2(4))(3)' OK
[1, 2, 3, None, 4] -> '1(2()(4))(3)' OK
[1] -> '1' OK
[-1, None, -2] -> '-1()(-2)' OK


## 10. Vì sao đúng

**Đúng theo cấu trúc.** Ta chứng minh bằng quy nạp theo **chiều cao** của cây rằng `dfs(node)` trả đúng chuỗi mà đề yêu cầu cho cây con gốc `node`.

- **Cơ sở:** `None` ⇒ chuỗi rỗng (nhóm ngoặc quanh nó, nếu có, thành `()`); lá ⇒ `str(val)` (đề: không con ⇒ không ngoặc).
- **Bước quy nạp:** giả sử `dfs(left)` và `dfs(right)` đúng cho các cây con.
  - Không có phải: đề bảo bọc con trái trong ngoặc và **bỏ** nhóm phải rỗng ⇒ `val(left)`.
  - Có phải: đề bảo viết nhóm trái rồi nhóm phải; nếu trái vắng thì **phải** có `()` ⇒ `val(L)(R)` với `L = ""` khi trái vắng, tức `()`. Đúng.

**Đúng theo "một-một" (không nhập nhằng).** Chuỗi kết quả có thể **đọc ngược** thành đúng cây ban đầu; ta kiểm bằng cách viết **trình phân tích ngược** `str2tree` và thử với mọi cây nhỏ (mục 15 giải thích chi tiết; cell dưới là bản gọn). Điều này chứng tỏ quy tắc "giữ `()` khi thiếu trái mà có phải" là **vừa đủ**: bỏ nó sẽ gộp hai cây khác nhau thành cùng một chuỗi.

In [19]:
def str2tree(s):
    """Đọc ngược chuỗi kết quả thành cây (ngăn xếp, không đệ quy). Chuỗi rỗng -> None."""
    if not s:
        return None
    i = 0
    n = len(s)

    def read_int(i):
        j = i
        if s[j] == "-":
            j += 1
        while j < n and s[j].isdigit():
            j += 1
        return int(s[i:j]), j

    val, i = read_int(0)
    root = TreeNode(val)
    stack = [[root, 0]]               # [nút, số nhóm ngoặc con đã gặp]
    while i < n:
        ch = s[i]
        if ch == "(":
            if s[i + 1] == ")":       # nhóm rỗng: con vắng
                stack[-1][1] += 1
                i += 2
            else:
                val, i = read_int(i + 1)
                child = TreeNode(val)
                parent, seen = stack[-1]
                if seen == 0:
                    parent.left = child
                else:
                    parent.right = child
                stack[-1][1] += 1
                stack.append([child, 0])
        else:                          # ")"
            stack.pop()
            i += 1
    return root

def tree_equal(a, b):
    return tree_to_list(a) == tree_to_list(b)

# Vòng khứ hồi: cây -> chuỗi -> cây phải trả lại đúng cây ban đầu, với nhiều cây ngẫu nhiên có từ 1 đến 7 nút (giá trị trong [-2, 2])
rng = random.Random(606)
count = 0
for n in range(1, 8):
    for _ in range(200):
        vals = random_tree_list(rng, n, -2, 2)
        t = build_tree(vals)
        s = Solution().tree2str(t)
        assert tree_to_list(str2tree(s)) == vals, (vals, s)
        count += 1
print("khứ hồi cây -> chuỗi -> cây đúng trên", count, "cây ngẫu nhiên: OK")

khứ hồi cây -> chuỗi -> cây đúng trên 1400 cây ngẫu nhiên: OK


**Đúng về kết thúc và độ phức tạp:** mỗi nút gọi `dfs` đúng một lần cho mỗi con thật (và một lần cho mỗi con `None`), nên số lời gọi tối đa `2n + 1`.

## 11. Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao (từ `log n` đến `n`).

| Cách | Time | Space |
|---|---|---|
| 1. Dựng đầy đủ + regex | `O(n)` (+ sao chép chuỗi lồng) | `O(n)` (chuỗi đầy đủ dài hơn) |
| 2. Đệ quy ghép chuỗi | `O(n)` lời gọi, tới `O(n²)` ký tự sao chép với cây một dây | `O(h)` ngăn xếp + chuỗi trung gian |
| 3. Đệ quy + list chung | `O(n)` | `O(n)` kết quả + `O(h)` ngăn xếp |
| 4. Ngăn xếp nút + ngoặc | `O(n)` | `O(n)` |
| 5. Ngăn xếp + tập đã thăm | `O(n)` | `O(n)` (thêm tập `visited`) |

Độ dài chuỗi kết quả là `Σ (độ dài chữ số/dấu) + 2·(số cặp ngoặc)`, với số cặp ngoặc = `(n - 1)` (mỗi nút không phải gốc có đúng một cặp bọc nó) cộng thêm số nút có phải mà không có trái (mỗi nút như vậy thêm một `()`). Ta đếm trực tiếp để xác nhận công thức này.

In [20]:
def predicted_length(root):
    digits = pairs = 0
    stack = [root]
    while stack:
        node = stack.pop()
        digits += len(str(node.val))
        if node.left is not None:
            stack.append(node.left)
            pairs += 1
        if node.right is not None:
            stack.append(node.right)
            pairs += 1
            if node.left is None:
                pairs += 1                      # thêm "()" cho nhóm trái vắng
    return digits + 2 * pairs

rng = random.Random(7)
for n in [1, 5, 50, 500, 5000]:
    t = build_tree(random_tree_list(rng, n, -1000, 1000))
    s = Solution().tree2str(t)
    assert predicted_length(t) == len(s)
    print(f"n = {n:5d}: độ dài chuỗi = {len(s):6d} = dự đoán {predicted_length(t):6d}")

n =     1: độ dài chuỗi =      4 = dự đoán      4
n =     5: độ dài chuỗi =     30 = dự đoán     30
n =    50: độ dài chuỗi =    281 = dự đoán    281
n =   500: độ dài chuỗi =   2868 = dự đoán   2868


n =  5000: độ dài chuỗi =  28721 = dự đoán  28721


## 12. Bộ test

Gom các cách vào một dict. Các phần: ví dụ đề, edge case, phản ví dụ (hai cây khác nhau không được ra cùng một chuỗi), khứ hồi bằng `str2tree`, ngẫu nhiên có seed đúng ràng buộc, và cây **một dây sâu `10^4`** (kiểm tra độ sâu).

In [21]:
methods = {
    "brute": lambda r: tree2str_bruteforce(r),
    "recursive": lambda r: SolutionRecursive().tree2str(r),
    "shared": lambda r: SolutionSharedList().tree2str(r),
    "tokens": lambda r: SolutionStackTokens().tree2str(r),
    "visited": lambda r: SolutionVisited().tree2str(r),
    "final": lambda r: Solution().tree2str(r),
}

def check_all(vals, expected=None):
    results = {name: fn(build_tree(vals)) for name, fn in methods.items()}
    assert len(set(results.values())) == 1, (vals, results)
    r = results["final"]
    if expected is not None:
        assert r == expected, (vals, r, expected)
    return r

sys.setrecursionlimit(40000)      # cây sâu 10^4 cần nâng giới hạn cho các cách đệ quy

# 1) Ví dụ đề
for vals, expected in examples:
    check_all(vals, expected)
print("ví dụ đề: OK")

# 2) Edge case
edge = [
    ([0], "0"),                                         # giá trị 0
    ([-1000], "-1000"),
    ([1, 2], "1(2)"),                                   # chỉ có con trái
    ([1, None, 2], "1()(2)"),                           # chỉ có con phải
    ([1, 2, 3], "1(2)(3)"),
    ([1, 2, None, 3], "1(2(3))"),                       # dây trái
    ([1, None, 2, None, 3], "1()(2()(3))"),             # dây phải
    ([1, 2, None, None, 3], "1(2()(3))"),               # trái rồi phải
    ([1, None, 2, 3], "1()(2(3))"),                     # phải rồi trái
    ([0, -1, 1], "0(-1)(1)"),
    ([1, 2, 3, 4, None, None, 5], "1(2(4))(3()(5))"),
    ([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1], "5(4(11(7)(2)))(8(13)(4()(1)))"),
]
for vals, expected in edge:
    check_all(vals, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


In [22]:
# 3) Phản ví dụ: hai cây KHÁC nhau phải cho hai chuỗi KHÁC nhau (nếu bỏ () khi thiếu trái, chúng bị nhập nhằng)
a = [1, 2, 3, 4]                 # 2 có con TRÁI là 4
b = [1, 2, 3, None, 4]           # 2 có con PHẢI là 4
sa, sb = check_all(a), check_all(b)
print(a, "->", sa)
print(b, "->", sb)
assert sa != sb

def tree2str_drop_all_empty(node):                 # phiên bản SAI: bỏ mọi ()
    if node is None:
        return ""
    if node.left is None and node.right is None:
        return str(node.val)
    parts = str(node.val)
    if node.left is not None:
        parts += "(" + tree2str_drop_all_empty(node.left) + ")"
    if node.right is not None:
        parts += "(" + tree2str_drop_all_empty(node.right) + ")"
    return parts

print("bản sai cho cả hai cây cùng chuỗi:", tree2str_drop_all_empty(build_tree(a)), tree2str_drop_all_empty(build_tree(b)))
assert tree2str_drop_all_empty(build_tree(a)) == tree2str_drop_all_empty(build_tree(b))      # nhập nhằng!

# 4) Tính một-một: mọi cây khác nhau (<= 4 nút) cho chuỗi khác nhau; khứ hồi bằng str2tree
seen = {}
rng = random.Random(1)
for n in range(1, 5):
    for _ in range(400):
        vals = random_tree_list(rng, n, 1, 2)
        s = check_all(vals)
        assert tree_to_list(str2tree(s)) == vals
        if s in seen:
            assert seen[s] == vals, (s, seen[s], vals)      # cùng chuỗi thì phải cùng cây
        seen[s] = vals
print("một-một + khứ hồi trên", len(seen), "chuỗi khác nhau: OK")

[1, 2, 3, 4] -> 1(2(4))(3)
[1, 2, 3, None, 4] -> 1(2()(4))(3)
bản sai cho cả hai cây cùng chuỗi: 1(2(4))(3) 1(2(4))(3)


một-một + khứ hồi trên 217 chuỗi khác nhau: OK


In [23]:
# 5) Ngẫu nhiên có seed, đúng ràng buộc: n <= 10^4, val trong [-1000, 1000]
rng = random.Random(2026)
for _ in range(300):
    n = rng.randint(1, 40)
    vals = random_tree_list(rng, n, -1000, 1000)
    s = check_all(vals)
    assert tree_to_list(str2tree(s)) == vals
print("ngẫu nhiên nhỏ (khứ hồi + 6 cách khớp nhau): OK")

for n in [200, 1000, 3000, 10000]:
    vals = random_tree_list(rng, n, -1000, 1000)
    s = check_all(vals)
    assert tree_to_list(str2tree(s)) == vals
    print(f"ngẫu nhiên n = {n:5d}: độ dài chuỗi {len(s)}")

# 6) Cây một dây sâu 10^4 (dựng trực tiếp, không qua list để khỏi tốn)
def chain(n, side):
    root = TreeNode(1)
    cur = root
    for i in range(2, n + 1):
        nxt = TreeNode((i % 2001) - 1000)
        if side == "left":
            cur.left = nxt
        else:
            cur.right = nxt
        cur = nxt
    return root

for side in ["left", "right"]:
    root = chain(10000, side)
    outs = {name: fn(root) for name, fn in methods.items()}
    assert len(set(outs.values())) == 1
    s = outs["final"]
    print(f"dây {side:5s} sâu 10000: độ dài chuỗi {len(s)} | đầu {s[:22]!r}")
    assert tree_to_list(str2tree(s)) == tree_to_list(root)

ngẫu nhiên nhỏ (khứ hồi + 6 cách khớp nhau): OK
ngẫu nhiên n =   200: độ dài chuỗi 1148
ngẫu nhiên n =  1000: độ dài chuỗi 5684


ngẫu nhiên n =  3000: độ dài chuỗi 17191


ngẫu nhiên n = 10000: độ dài chuỗi 57217


dây left  sâu 10000: độ dài chuỗi 53912 | đầu '1(-998(-997(-996(-995('


dây right sâu 10000: độ dài chuỗi 73910 | đầu '1()(-998()(-997()(-996'


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bỏ mọi `()` rỗng (kể cả khi thiếu con trái mà có con phải)

Đọc nhanh đề "bỏ các cặp `()` rỗng" rồi bỏ **tất cả** — quên ngoại lệ.

In [24]:
class WrongDropAll:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        def dfs(node):
            if node is None:
                return ""
            s = str(node.val)
            if node.left is not None:
                s += "(" + dfs(node.left) + ")"
            if node.right is not None:
                s += "(" + dfs(node.right) + ")"
            return s
        return dfs(root)

t = build_tree([1, 2, 3, None, 4])
print("sai :", WrongDropAll().tree2str(t), "(đúng phải là 1(2()(4))(3))")
print("đúng:", Solution().tree2str(t))

sai : 1(2(4))(3) (đúng phải là 1(2()(4))(3))
đúng: 1(2()(4))(3)


**Vì sao sai:** `1(2(4))(3)` không phân biệt được `4` là con trái hay con phải của `2`. Đề yêu cầu giữ `()` đứng trước con phải khi con trái vắng. **Cách sửa:** nếu có con phải thì luôn in nhóm trái `(...)`, kể cả khi trống.

### Lỗi 2: giữ mọi `()` (không bỏ ngoặc rỗng thừa)

In [25]:
class WrongKeepAll:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        def dfs(node):
            if node is None:
                return ""
            return f"{node.val}({dfs(node.left)})({dfs(node.right)})"
        return dfs(root)

t = build_tree([1, 2, 3, 4])
print("sai :", WrongKeepAll().tree2str(t), "(đúng phải là 1(2(4))(3))")
print("đúng:", Solution().tree2str(t))

sai : 1(2(4()())())(3()()) (đúng phải là 1(2(4))(3))
đúng: 1(2(4))(3)


**Vì sao sai:** đề nói rõ bỏ các `()` thừa (lá không con, nút chỉ có con trái không cần nhóm phải rỗng). **Cách sửa:** thêm hai nhánh kiểm tra `lá` và `không có phải` như mục 6.

### Lỗi 3: quên `str()` khi nối giá trị

In [26]:
class WrongNoStr:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        def dfs(node):
            if node is None:
                return ""
            if node.left is None and node.right is None:
                return node.val                          # SAI: trả số nguyên, không phải chuỗi
            if node.right is None:
                return node.val + "(" + dfs(node.left) + ")"   # SAI: int + str
            return node.val + "(" + dfs(node.left) + ")(" + dfs(node.right) + ")"
        return dfs(root)

try:
    WrongNoStr().tree2str(build_tree([1, 2, 3]))
except TypeError as e:
    print("lỗi TypeError:", e)
print("đúng:", Solution().tree2str(build_tree([1, 2, 3])))

lỗi TypeError: unsupported operand type(s) for +: 'int' and 'str'
đúng: 1(2)(3)


**Vì sao sai:** `val` là số nguyên; `int + str` không được phép. Lá trả về số thay vì chuỗi cũng làm kiểu trả về sai. **Cách sửa:** `str(node.val)` hoặc f-string (`f"{node.val}(...)"`), kể cả cho số âm.

### Lỗi 4: đệ quy sâu quá giới hạn mặc định (cây một dây `10^4` nút)

In [27]:
def naive_dfs(node):
    if node is None:
        return ""
    if node.left is None and node.right is None:
        return str(node.val)
    if node.right is None:
        return f"{node.val}({naive_dfs(node.left)})"
    return f"{node.val}({naive_dfs(node.left)})({naive_dfs(node.right)})"

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)                       # giới hạn mặc định
deep = chain(3000, "left")
try:
    naive_dfs(deep)
    print("không lỗi")
except RecursionError as e:
    print("sai : RecursionError:", str(e)[:60])
print("đúng (ngăn xếp tự quản lý):", len(SolutionStackTokens().tree2str(deep)), "ký tự")
sys.setrecursionlimit(old)

sai : RecursionError: maximum recursion depth exceeded
đúng (ngăn xếp tự quản lý): 16670 ký tự


**Vì sao sai:** mỗi tầng đệ quy chiếm một khung ngăn xếp; vượt `1000` tầng Python ném `RecursionError`. **Cách sửa:** (1) nâng `sys.setrecursionlimit` (cẩn thận với tràn ngăn xếp thật), hoặc (2) dùng **ngăn xếp tự quản lý** như cách 4 / cách 5 — an toàn với mọi độ sâu.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "cây → chuỗi / duyệt theo thứ tự":**

- Đề nói **preorder / inorder / postorder** hay "biểu diễn cây bằng chuỗi" ⇒ nghĩ đệ quy DFS: **in nút, rồi con**.
- Cần **biến thể tại chỗ trống** (bỏ hay giữ `()` khi thiếu con): hãy **liệt kê bảng** `(left, right)` → chuỗi (như mục 6) rồi gộp trường hợp trùng.
- Để chuỗi đọc ngược lại được cây, mọi nhánh khuyết **không thể bị bỏ qua lặng lẽ** — mất thông tin vị trí.

**Mẹo nhỏ:**

- `dfs(None)` trả `""` giúp nhóm trái rỗng **tự thành `()`**, không cần nhánh riêng.
- Gom mảnh vào `list` rồi `"".join` một lần, tránh nối chuỗi `O(n²)`.
- Với cây có thể sâu: ưu tiên **ngăn xếp tự quản lý**, hoặc nâng giới hạn đệ quy.
- Dùng `is None` thay vì `if node.val:` (vì `0` là "sai").

**Họ bài liên quan:**

| Bài | Ý tưởng |
|---|---|
| 536. Construct Binary Tree from String | làm ngược lại: parse chuỗi ngoặc thành cây |
| 297. Serialize and Deserialize Binary Tree | chuỗi hóa với dấu `null`, hai chiều |
| 144. Binary Tree Preorder Traversal | preorder cơ bản (đệ quy / ngăn xếp) |
| 652. Find Duplicate Subtrees | chuỗi hóa cây con để so sánh / băm |
| 572. Subtree of Another Tree | so sánh cây qua chuỗi hóa hay so khớp |

## 15. Biến thể

### 15.1. Làm ngược lại: từ chuỗi dựng lại cây (LeetCode 536)

Ta đã dùng `str2tree` để kiểm tra khứ hồi. Đây chính là bài 536: "cho chuỗi dạng ngoặc, dựng cây". Cách làm: duyệt từng ký tự với **ngăn xếp** các nút, mỗi nút nhớ đã gặp bao nhiêu nhóm ngoặc con (0 ⇒ nhóm kế là con trái, 1 ⇒ con phải); nhóm `()` rỗng chỉ tăng bộ đếm.

In [28]:
for s in ["1(2(4))(3)", "1(2()(4))(3)", "-1()(-2)", "7", "4(2(3)(1))(6(5))"]:
    t = str2tree(s)
    back = Solution().tree2str(t)
    assert back == s, (s, back)
    print(f"{s:22s} -> cây {tree_to_list(t)} -> chuỗi {back}")

1(2(4))(3)             -> cây [1, 2, 3, 4] -> chuỗi 1(2(4))(3)
1(2()(4))(3)           -> cây [1, 2, 3, None, 4] -> chuỗi 1(2()(4))(3)
-1()(-2)               -> cây [-1, None, -2] -> chuỗi -1()(-2)
7                      -> cây [7] -> chuỗi 7
4(2(3)(1))(6(5))       -> cây [4, 2, 6, 3, 1, 5] -> chuỗi 4(2(3)(1))(6(5))


### 15.2. Chuỗi đầy đủ (không bỏ `()` nào)

Phiên bản "gốc" trước khi lược bớt: mỗi nút kèm đủ hai nhóm ngoặc (như `full_form` ở mục 4). Vì mỗi nút kèm đúng hai nhóm ngoặc nên chuỗi đầy đủ có đúng `2n` cặp ngoặc, tức độ dài là `(số ký tự các giá trị) + 4n`. Kiểm bằng code, và đo mức tiết kiệm của quy tắc rút gọn.

In [29]:
rng = random.Random(15)
for n in [1, 3, 10, 100, 1000]:
    t = build_tree(random_tree_list(rng, n, 0, 9))      # chữ số đơn để dễ đếm
    full = full_form(t)
    short = Solution().tree2str(t)
    assert full.count("(") == 2 * n                    # mỗi nút kèm đúng 2 nhóm ngoặc
    print(f"n = {n:4d}: đầy đủ {len(full):5d} ký tự | rút gọn {len(short):5d} | tiết kiệm {100 * (1 - len(short) / len(full)):.1f}%")

n =    1: đầy đủ     5 ký tự | rút gọn     1 | tiết kiệm 80.0%
n =    3: đầy đủ    15 ký tự | rút gọn     7 | tiết kiệm 53.3%
n =   10: đầy đủ    50 ký tự | rút gọn    28 | tiết kiệm 44.0%
n =  100: đầy đủ   500 ký tự | rút gọn   332 | tiết kiệm 33.6%
n = 1000: đầy đủ  5000 ký tự | rút gọn  3330 | tiết kiệm 33.4%


### 15.3. Dự đoán độ dài chuỗi mà không dựng nó

Độ dài = `Σ len(str(val))` + `2 × (số nút khác gốc)` + `2 × (số nút có con phải nhưng không có con trái)`. Ta dùng công thức này ở mục 11; ở đây làm thành hàm gọn và kiểm bằng nhiều cây.

In [30]:
def length_without_building(root):
    total = 0
    stack = [root]
    while stack:
        node = stack.pop()
        total += len(str(node.val))
        for child in (node.left, node.right):
            if child is not None:
                total += 2                    # cặp ngoặc bọc con thật
                stack.append(child)
        if node.left is None and node.right is not None:
            total += 2                        # "()" thay cho con trái vắng
    return total

rng = random.Random(33)
for _ in range(200):
    n = rng.randint(1, 80)
    t = build_tree(random_tree_list(rng, n, -1000, 1000))
    assert length_without_building(t) == len(Solution().tree2str(t))
print("công thức độ dài khớp trên 200 cây ngẫu nhiên: OK")

công thức độ dài khớp trên 200 cây ngẫu nhiên: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đầy đủ + regex | dựng chuỗi có đủ `()` rồi xóa `()` thừa | `O(n)` (+ sao chép) | `O(n)` lớn hơn | Chỉ để làm oracle độc lập |
| 2. Đệ quy 3 trường hợp | mỗi nút trả chuỗi, cha ghép | `O(n)` lời gọi, `O(n²)` ký tự với dây dài | `O(h)` ngăn xếp | **Nộp bài**: ngắn, đúng đề |
| 3. Đệ quy + list chung | mọi tầng ghi vào `parts` | `O(n)` | `O(n) + O(h)` | Khi muốn tuyến tính chặt, vẫn đệ quy |
| 4. Ngăn xếp nút + ngoặc | đẩy ngược nhóm phải rồi nhóm trái | `O(n)` | `O(n)` | Cây rất sâu (không đệ quy), dễ hiểu |
| 5. Ngăn xếp + `visited` | mở nút lần đầu, đóng lần hai | `O(n)` | `O(n)` + tập | Khi quen mẫu "thăm hai lần" của DFS lặp |

Đo thời gian thực tế trên cây ngẫu nhiên lớn và cây một dây (số liệu chỉ để so sánh tương đối).

In [31]:
rng = random.Random(3)
rand_tree = build_tree(random_tree_list(rng, 10000, -1000, 1000))
trees = {"ngẫu nhiên 10^4": rand_tree, "dây trái 10^4": chain(10000, "left"), "dây phải 10^4": chain(10000, "right")}
for tname, root in trees.items():
    ref = None
    for mname, fn in methods.items():
        t0 = time.perf_counter()
        s = fn(root)
        dt = time.perf_counter() - t0
        ref = s if ref is None else ref
        assert s == ref
        print(f"{tname:16s} {mname:10s} {dt:7.4f}s")
    print()

ngẫu nhiên 10^4  brute       0.0055s
ngẫu nhiên 10^4  recursive   0.0030s
ngẫu nhiên 10^4  shared      0.0025s
ngẫu nhiên 10^4  tokens      0.0040s
ngẫu nhiên 10^4  visited     0.0036s
ngẫu nhiên 10^4  final       0.0023s



dây trái 10^4    brute       0.0152s
dây trái 10^4    recursive   0.0114s
dây trái 10^4    shared      0.0034s
dây trái 10^4    tokens      0.0049s
dây trái 10^4    visited     0.0041s
dây trái 10^4    final       0.0112s



dây phải 10^4    brute       0.0152s
dây phải 10^4    recursive   0.0152s
dây phải 10^4    shared      0.0038s


dây phải 10^4    tokens      0.0080s
dây phải 10^4    visited     0.0044s


dây phải 10^4    final       0.0157s



## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Các cell `TODO` chỉ **định nghĩa** hàm, không tự gọi, để notebook luôn chạy sạch.

### Bài 1 (đoán output)

Không chạy máy, dự đoán chuỗi `tree2str` của 4 cây rồi đối chiếu đáp án ở mục 19:

```python
[1, 2]
[1, None, 2]
[1, 2, None, 3]
[1, 2, 3]
```

### Bài 2 (tính tay)

Với cây `[1,2,3,4,None,None,5]` (gốc `1`; `2` có con trái `4`; `3` có con phải `5`), hãy viết chuỗi **đầy đủ** (mỗi nút kèm đủ hai nhóm ngoặc) rồi chuỗi **rút gọn** theo đề.

### Bài 3 (điền TODO)

Hoàn thành hàm đệ quy: điền **ba nhánh** `if` còn thiếu.

In [32]:
def tree2str_fill_todo(root):
    def dfs(node):
        if node is None:
            return ""
        if node.left is None and node.right is None:
            pass  # TODO: trường hợp lá
        if node.right is None:
            pass  # TODO: không có con phải
        # TODO: có con phải (trái có thể vắng)
        return None
    return dfs(root)
# Không gọi tree2str_fill_todo ở đây. Hãy sửa rồi tự gọi ở một cell mới.

### Bài 4 (tự viết lại)

Không nhìn lại lời giải, hãy viết `tree2str_iterative(root)` **không dùng đệ quy**, theo cách ngăn xếp chứa cả nút lẫn ký tự ngoặc (cách 4). Nhớ thứ tự đẩy vào ngăn xếp là **ngược** thứ tự in ra.

In [33]:
def tree2str_iterative(root):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

### Bài 5 (mini challenge)

Viết `height_from_string(s)` tính **chiều cao của cây** (số nút trên đường dài nhất từ gốc xuống lá) **trực tiếp từ chuỗi** `s` mà **không dựng cây**. Gợi ý: độ sâu lồng nhau lớn nhất của ngoặc liên quan thế nào tới chiều cao? Kiểm chứng với chiều cao tính trên cây thật.

In [34]:
def height_from_string(s):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: nút có con trái duy nhất → `x(trái)`; nút chỉ có con phải → phải giữ `()`.
- Mức 2: `[1,2]` là `1` với con trái `2`; `[1,None,2]` là `1` với con phải `2`.
- Mức 3: đáp án `"1(2)"`, `"1()(2)"`, `"1(2(3))"`, `"1(2)(3)"` (chú ý `[1,2,None,3]`: `1` có con trái `2` và không có con phải; `3` là con trái của `2`).

**Bài 2**

- Mức 1: mỗi nút kèm hai nhóm ngoặc; con vắng là `()`.
- Mức 2: `4` và `5` là lá: `4()()`, `5()()`; nút `2` = `2(4()())()`; nút `3` = `3()(5()())`.
- Mức 3: rút gọn: `2(4)`, `3()(5)`; cả cây `1(2(4))(3()(5))`.

**Bài 3**

- Mức 1: lá ⇒ chỉ `str(val)`.
- Mức 2: không có phải ⇒ bọc trái; có phải ⇒ bọc cả hai, dùng `dfs(None) = ""`.
- Mức 3: `return str(node.val)` / `return f"{node.val}({dfs(node.left)})"` / `return f"{node.val}({dfs(node.left)})({dfs(node.right)})"` (nhớ thêm `if` kiểm tra đúng thứ tự).

**Bài 4**

- Mức 1: ngăn xếp chứa nút hoặc chuỗi ký tự; chuỗi thì ghi thẳng.
- Mức 2: đẩy nhóm **phải** trước rồi nhóm **trái**; mỗi nhóm đẩy theo thứ tự `")"`, nút, `"("`.
- Mức 3: trái vắng mà có phải thì đẩy `")", "("` để in `()`; lá không đẩy gì.

**Bài 5**

- Mức 1: mỗi nút không phải gốc nằm trong một cặp ngoặc bọc nó.
- Mức 2: độ sâu ngoặc cực đại = độ sâu lớn nhất của nút (gốc độ sâu 0); ngoặc rỗng `()` không vượt quá độ sâu của một nút thật.
- Mức 3: duyệt ký tự, `depth += 1` khi gặp `(`, `depth -= 1` khi gặp `)`, nhớ cực đại; chiều cao = cực đại + 1.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1
answers1 = {
    (1, 2): "1(2)",
    (1, None, 2): "1()(2)",
    (1, 2, None, 3): "1(2(3))",
    (1, 2, 3): "1(2)(3)",
}
for vals, expected in answers1.items():
    got = Solution().tree2str(build_tree(list(vals)))
    assert got == expected, (vals, got, expected)
    print(list(vals), "->", got)

# Bài 2
t = build_tree([1, 2, 3, 4, None, None, 5])
print("Bài 2: đầy đủ  :", full_form(t))
print("Bài 2: rút gọn :", Solution().tree2str(t))
assert Solution().tree2str(t) == "1(2(4))(3()(5))"
assert full_form(t) == "1(2(4()())())(3()(5()()))"

[1, 2] -> 1(2)
[1, None, 2] -> 1()(2)
[1, 2, None, 3] -> 1(2(3))
[1, 2, 3] -> 1(2)(3)
Bài 2: đầy đủ  : 1(2(4()())())(3()(5()()))
Bài 2: rút gọn : 1(2(4))(3()(5))


In [36]:
# Bài 3
def tree2str_fill_answer(root):
    def dfs(node):
        if node is None:
            return ""
        if node.left is None and node.right is None:
            return str(node.val)
        if node.right is None:
            return f"{node.val}({dfs(node.left)})"
        return f"{node.val}({dfs(node.left)})({dfs(node.right)})"
    return dfs(root)

# Bài 4
def tree2str_iterative_answer(root):
    res = []
    stack = [root]
    while stack:
        x = stack.pop()
        if isinstance(x, str):
            res.append(x)
        elif x is not None:
            res.append(str(x.val))
            if x.right is not None:
                stack.extend([")", x.right, "("])
            if x.left is not None:
                stack.extend([")", x.left, "("])
            elif x.right is not None:
                stack.extend([")", "("])
    return "".join(res)

# Bài 5
def height_from_string_answer(s):
    depth = best = 0
    for ch in s:
        if ch == "(":
            depth += 1
            best = max(best, depth)
        elif ch == ")":
            depth -= 1
    return best + 1

def real_height(root):
    h, level = 0, [root]
    while level:
        h += 1
        level = [c for node in level for c in (node.left, node.right) if c is not None]
    return h

rng = random.Random(1999)
for _ in range(300):
    n = rng.randint(1, 60)
    vals = random_tree_list(rng, n, -1000, 1000)
    root = build_tree(vals)
    expected = Solution().tree2str(root)
    assert tree2str_fill_answer(root) == expected
    assert tree2str_iterative_answer(root) == expected
    assert height_from_string_answer(expected) == real_height(root), vals
print("Bài 3, 4, 5: OK")

Bài 3, 4, 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đệ quy 3 trường hợp):**

```python
class Solution:
    def tree2str(self, root: Optional[TreeNode]) -> str:
        def dfs(node):
            if node is None:
                return ""                                   # con vắng -> chuỗi rỗng -> nhóm "()" tự xuất hiện
            if node.left is None and node.right is None:
                return str(node.val)                        # lá: chỉ giá trị
            if node.right is None:
                return f"{node.val}({dfs(node.left)})"      # chỉ có trái: bỏ nhóm phải
            return f"{node.val}({dfs(node.left)})({dfs(node.right)})"   # có phải: giữ cả hai nhóm
        return dfs(root)
```

**Câu thần chú:** *"Lá: chỉ số. Không có phải: bọc trái. Có phải: bọc cả hai, trái vắng thì `()`."*

**Checklist trước khi submit:**

- [ ] Hiểu quy tắc **không đối xứng**: thiếu phải ⇒ bỏ `()`, thiếu trái mà có phải ⇒ **giữ** `()`.
- [ ] Dùng `str(val)` / f-string (không cộng `int` với `str`), đúng cho số âm và `0`.
- [ ] Kiểm tra lá **trước**, rồi "không có phải", rồi mới đến trường hợp đầy đủ.
- [ ] `dfs(None)` trả `""`, không trả `None`.
- [ ] Cây có thể sâu `10^4`: cân nhắc nâng giới hạn đệ quy hoặc dùng ngăn xếp tự quản lý.
- [ ] Đã thử: một nút, nút giá trị `0` / âm, dây trái, dây phải, chỉ có con phải.
- [ ] Đã kiểm khứ hồi bằng `str2tree` (chuỗi ⇒ cây ⇒ chuỗi) trên cây ngẫu nhiên nhỏ.
- [ ] Độ phức tạp: `O(n)` thời gian; bộ nhớ `O(n)` kết quả + `O(h)` ngăn xếp.